# CodeAlpha Internship – Task 3
# Car Price Prediction using Machine Learning

**Intern:** Deepika B.  
**Domain:** Data Science  
**Task:** Car Price Prediction with Machine Learning  
**Model:** Random Forest Regressor  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn

## Project objective
Build a regression model that estimates a used car's selling price from available car details such as year, kilometres driven, fuel, transmission, mileage, engine capacity, maximum power and other available attributes.

**Important dataset note:** The CodeAlpha task sheet asks for car-related features such as brand, horsepower and mileage, but it does not specify one exact dataset URL or schema. This notebook is designed to work with a CarDekho-style CSV. Upload the CSV you download from the dataset source. The notebook will inspect the columns and use the compatible features it finds. It will not fabricate car records.

## How to use
1. Download the CarDekho vehicle CSV from the Kaggle dataset page linked in the instructions.
2. In Google Colab, run the dataset upload cell and select the CSV file.
3. Run the remaining cells from top to bottom.


## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully!")

## 2. Upload and load the dataset

For Google Colab, choose the CSV file downloaded from Kaggle. If running locally, place the CSV beside this notebook and enter its filename when prompted.


In [ ]:
# Google Colab upload
try:
    from google.colab import files
    uploaded = files.upload()
    csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
    if not csv_files:
        raise ValueError("No CSV file was uploaded. Please upload a .csv dataset.")
    DATA_PATH = csv_files[0]
except ImportError:
    DATA_PATH = input("Enter the path/name of your CSV file: ").strip()

df = pd.read_csv(DATA_PATH)
print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())

## 3. Understand and inspect the dataset

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing values"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nStatistical summary:")
display(df.describe(include="all").T)

## 4. Clean column names and identify the target

The target is the selling price. The code accepts common names such as `selling_price`, `price`, or `sellingprice`. It also standardizes column names to make processing easier.


In [ ]:
def clean_column_name(column):
    column = str(column).strip().lower()
    column = re.sub(r"[^a-z0-9]+", "_", column)
    return column.strip("_")

df.columns = [clean_column_name(c) for c in df.columns]

target_candidates = ["selling_price", "sellingprice", "price", "sale_price"]
target_col = next((c for c in target_candidates if c in df.columns), None)

if target_col is None:
    raise ValueError(
        "Could not identify the selling-price column. "
        f"Available columns: {df.columns.tolist()}. "
        "Rename the target column to 'selling_price' and rerun this cell."
    )

# Convert target to numeric and remove rows where target is missing/invalid.
df[target_col] = pd.to_numeric(
    df[target_col].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)
df = df.dropna(subset=[target_col]).copy()
df = df[df[target_col] > 0].copy()

# Remove exact duplicate rows.
df = df.drop_duplicates().reset_index(drop=True)

print("Target column:", target_col)
print("Cleaned dataset shape:", df.shape)
display(df.head())

## 5. Feature engineering

CarDekho-style datasets sometimes store mileage, engine capacity and maximum power as text with units (for example, `23.4 kmpl`, `1248 CC`, or `74 bhp`). This step extracts the numeric part when those columns are present. It also creates car age from the year column when available.


In [ ]:
# Convert common measurement columns from text-with-units to numeric.
measurement_columns = ["mileage", "engine", "max_power", "maxpower", "horsepower", "km_driven", "year", "seats"]

for col in measurement_columns:
    if col in df.columns:
        extracted = df[col].astype(str).str.replace(",", "", regex=False).str.extract(r"([-+]?[0-9]*\.?[0-9]+)")[0]
        numeric_values = pd.to_numeric(extracted, errors="coerce")
        # Use converted values if at least one valid number exists.
        if numeric_values.notna().any():
            df[col] = numeric_values

if "year" in df.columns:
    current_year = pd.Timestamp.today().year
    df["car_age"] = (current_year - df["year"]).clip(lower=0)

print("Feature engineering completed.")
print("Available columns:", df.columns.tolist())
display(df.head())

## 6. Exploratory Data Analysis (EDA)

The following graphs help understand the distribution of car prices and relationships between price and available numerical features.


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df[target_col], kde=True)
plt.title("Distribution of Car Selling Prices")
plt.xlabel("Selling Price")
plt.ylabel("Number of Cars")
plt.tight_layout()
plt.show()

In [ ]:
numeric_df = df.select_dtypes(include=np.number)

if target_col in numeric_df.columns and numeric_df.shape[1] >= 2:
    correlations = numeric_df.corr(numeric_only=True)
    plt.figure(figsize=(10, 7))
    sns.heatmap(correlations, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Correlation Between Numerical Features")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numerical columns for a correlation heatmap.")

In [ ]:
# Plot selling price against year or kilometres driven when available.
for feature in ["year", "car_age", "km_driven", "mileage", "engine", "max_power"]:
    if feature in df.columns and pd.api.types.is_numeric_dtype(df[feature]):
        plt.figure(figsize=(7, 4))
        sns.scatterplot(data=df, x=feature, y=target_col, alpha=0.6)
        plt.title(f"{feature.replace('_', ' ').title()} vs Selling Price")
        plt.tight_layout()
        plt.show()

## 7. Prepare features and target

The target (`y`) is the selling price. All other columns are candidate input features. Identifier-like columns, if present, are removed. Categorical features are one-hot encoded and missing values are imputed inside a Scikit-learn pipeline.


In [ ]:
# Remove columns that are usually identifiers or duplicates of the target.
drop_candidates = [c for c in ["car_id", "id", "unnamed_0", "selling_price", "sellingprice", "price", "sale_price"] if c in df.columns]
X = df.drop(columns=drop_candidates, errors="ignore")
y = df[target_col].copy()

# Remove columns that contain only one distinct non-null value.
constant_cols = [c for c in X.columns if X[c].nunique(dropna=True) <= 1]
X = X.drop(columns=constant_cols)

if X.shape[1] == 0:
    raise ValueError("No usable input features remain after cleaning.")

# Ensure numeric-looking columns are numeric where possible.
for col in X.columns:
    if X[col].dtype == "object":
        converted = pd.to_numeric(X[col], errors="coerce")
        if converted.notna().mean() >= 0.8:
            X[col] = converted

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Input features:", X.columns.tolist())
print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)
print("Number of records:", len(X))

## 8. Split data into training and testing sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

## 9. Build the preprocessing and regression pipeline

A **Random Forest Regressor** combines predictions from multiple decision trees. It can model non-linear relationships between car features and prices.


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)
print("Random Forest regression model trained successfully!")

## 10. Predict and evaluate the model

We evaluate the model on the held-out test set using:
- **MAE:** average absolute prediction error (lower is better).
- **RMSE:** gives larger errors more weight (lower is better).
- **R² score:** proportion of target variation explained by the model (closer to 1 is generally better).


In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:,.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:,.2f}")
print(f"R-squared (R2) Score: {r2:.4f}")

## 11. Actual vs predicted prices

In [ ]:
comparison = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": y_pred
})
display(comparison.head(10))

plt.figure(figsize=(7, 5))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.6)
min_value = min(float(y_test.min()), float(np.min(y_pred)))
max_value = max(float(y_test.max()), float(np.max(y_pred)))
plt.plot([min_value, max_value], [min_value, max_value], "r--")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Car Prices")
plt.tight_layout()
plt.show()

## 12. Predict the price of a new car

This cell creates an example input using the first test record, then predicts its price. This guarantees the example uses the same input columns and formats as the training data. You can later replace the values with another row of car details.


In [ ]:
sample_car = X_test.iloc[[0]].copy()
predicted_price = model.predict(sample_car)[0]

print("Example car details:")
display(sample_car)
print(f"Estimated selling price: {predicted_price:,.2f}")
print(f"Actual selling price in test data: {y_test.iloc[0]:,.2f}")

## 13. Conclusion

This project demonstrates a machine-learning regression workflow for estimating used-car selling prices. The dataset was inspected and cleaned, measurement fields were converted where possible, and a car-age feature was created when a year column was available. Numerical and categorical inputs were preprocessed using a Scikit-learn pipeline. A Random Forest Regressor was trained and evaluated using MAE, RMSE and R².

**Result interpretation:** The actual metric values depend on the dataset uploaded and its train/test split. Do not claim a particular accuracy or error value until the notebook has been run with the selected CSV.

### Limitations
- Predicted prices depend on the quality, coverage and date of the dataset.
- Important factors such as local demand, vehicle condition, accident history and location may be missing.
- This model is an educational estimate, not a professional vehicle valuation.

---
**CodeAlpha Internship – Task 3: Car Price Prediction**  
**Intern: Deepika B.**
